In [6]:
import pandas as pd
import numpy as np
from collections import Counter
import math

# =========================================================
# 1. LOAD DATA HASIL LABELING
# =========================================================
file_path = "hasil_labeling.csv"
df = pd.read_csv(file_path, sep=";", encoding="utf-8-sig")

# =========================================================
# 2. AMBIL KOLOM TEKS DAN LABEL
# =========================================================
df = df[["hasil_preprocessing", "label"]].copy()

# bersihkan data kosong
df["hasil_preprocessing"] = df["hasil_preprocessing"].astype(str).str.strip()
df = df[df["hasil_preprocessing"] != ""]
df = df.dropna(subset=["hasil_preprocessing"])

print("Jumlah data:", len(df))

# =========================================================
# 3. TOKENISASI UNIGRAM + BIGRAM
# =========================================================
def generate_terms(text):
    words = str(text).split()

    unigram = words
    bigram = [" ".join(words[i:i+2]) for i in range(len(words)-1)]

    return unigram + bigram

documents = df["hasil_preprocessing"].tolist()
labels = df["label"].tolist()

tokenized_docs = [generate_terms(doc) for doc in documents]

# =========================================================
# 4. HITUNG TF
# TF = jumlah kemunculan kata dalam dokumen
# =========================================================
tf_list = []
all_terms = set()

for tokens in tokenized_docs:
    term_count = Counter(tokens)
    tf_list.append(term_count)
    all_terms.update(term_count.keys())

all_terms = sorted(all_terms)

# =========================================================
# 5. HITUNG DF
# df = jumlah dokumen yang mengandung term
# =========================================================
df_dict = {}
for term in all_terms:
    df_dict[term] = sum(1 for doc in tokenized_docs if term in doc)

# =========================================================
# 6. HITUNG IDF
# IDF = log10(D / df)
# =========================================================
D = len(documents)

idf_dict = {}
for term in all_terms:
    if df_dict[term] > 0:
        idf_dict[term] = math.log10(D / df_dict[term])
    else:
        idf_dict[term] = 0

# =========================================================
# 7. HITUNG BOBOT TF-IDF
# W = TF * IDF
# =========================================================
tfidf_matrix = []

for term in all_terms:
    row = {
        "token": term,
        "df": df_dict[term],
        "idf": round(idf_dict[term], 6)
    }

    for i, tf_doc in enumerate(tf_list):
        tf_value = tf_doc.get(term, 0)
        w_value = tf_value * idf_dict[term]

        row[f"tf_D{i+1}"] = tf_value
        row[f"W_D{i+1}"] = round(w_value, 6)

    tfidf_matrix.append(row)

tfidf_table = pd.DataFrame(tfidf_matrix)

# =========================================================
# 8. SIMPAN TABEL TF-IDF MODEL TABEL SKRIPSI
# =========================================================
tfidf_table.to_csv("tabel_tfidf_manual_bigram.csv", sep=";", index=False, encoding="utf-8-sig")

print("\nFile tabel TF-IDF manual berhasil disimpan:")
print("- tabel_tfidf_manual_bigram.csv")

# =========================================================
# 9. UBAH MENJADI FITUR PER DOKUMEN
# format: baris = dokumen, kolom = term, isi = bobot TF-IDF
# =========================================================
fitur_dokumen = []

for i, tf_doc in enumerate(tf_list):
    row = {}
    for term in all_terms:
        tf_value = tf_doc.get(term, 0)
        row[term] = round(tf_value * idf_dict[term], 6)

    row["label"] = labels[i]
    fitur_dokumen.append(row)

tfidf_features = pd.DataFrame(fitur_dokumen)

# =========================================================
# 10. SIMPAN HASIL FITUR TF-IDF
# =========================================================
tfidf_features.to_csv("hasil_tfidf_manual_bigram.csv", sep=";", index=False, encoding="utf-8-sig")

print("\nFile fitur TF-IDF berhasil disimpan:")
print("- hasil_tfidf_manual_bigram.csv")

# =========================================================
# 11. TAMPILKAN CONTOH HASIL
# =========================================================
print("\nContoh tabel TF-IDF manual:")
print(tfidf_table.head(10))

print("\nContoh fitur TF-IDF per dokumen:")
print(tfidf_features.head())

Jumlah data: 1127

File tabel TF-IDF manual berhasil disimpan:
- tabel_tfidf_manual_bigram.csv

File fitur TF-IDF berhasil disimpan:
- hasil_tfidf_manual_bigram.csv

Contoh tabel TF-IDF manual:
          token  df       idf  tf_D1  W_D1  tf_D2  W_D2  tf_D3  W_D3  tf_D4  \
0             a  39  1.460859      0   0.0      0   0.0      0   0.0      0   
1           a e  36  1.495621      0   0.0      0   0.0      0   0.0      0   
2      a pimpin   1  3.051924      0   0.0      0   0.0      0   0.0      0   
3      a thread   1  3.051924      0   0.0      0   0.0      0   0.0      0   
4          aave   6  2.273773      0   0.0      0   0.0      0   0.0      0   
5   aave jaring   1  3.051924      0   0.0      0   0.0      0   0.0      0   
6     aave near   3  2.574803      0   0.0      0   0.0      0   0.0      0   
7   aave terima   1  3.051924      0   0.0      0   0.0      0   0.0      0   
8  aave toncoin   1  3.051924      0   0.0      0   0.0      0   0.0      0   
9          abad 